# Loading Waymo frames with PyTorch

This notebook opens a **converted full-frame** Waymo dataset and turns camera images, LiDAR points, range images, and annotations into PyTorch tensors. It works with both v2 Parquet-derived `.pkl.gz` caches and legacy v1 TFRecord-derived `.pkl` caches.

Set `TORCH_WAYMO_DATASET` to the converted root before starting Jupyter, or edit `dataset_root` below. See the project README for one-segment download and conversion commands.

In [ ]:
import io
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from PIL import Image
from torch.utils.data import DataLoader

from torch_waymo import WaymoDataset

In [ ]:
dataset_root = Path(
    os.environ.get(
        "TORCH_WAYMO_DATASET",
        "~/Datasets/Waymo-v2-subset/converted",
    )
).expanduser()
dataset = WaymoDataset(dataset_root, "training")
frame = dataset[0]

assert hasattr(frame, "images"), "Use the full converted cache, not converted_simplified."
print(f"{len(dataset)} frames")
print(f"segment: {frame.context.name}")
print(f"timestamp: {frame.timestamp_micros}")
print(f"cameras={len(frame.images)}, lasers={len(frame.lasers)}, 3D labels={len(frame.laser_labels)}")

## Camera image and 2D annotations

Images remain JPEG-compressed in the cache. Decode only the camera needed by a training sample, then convert the HWC uint8 array to a CHW PyTorch tensor.

In [ ]:
camera = frame.images[0]
camera_array = np.array(Image.open(io.BytesIO(camera.image)).convert("RGB"), copy=True)
camera_tensor = torch.from_numpy(camera_array).permute(2, 0, 1)
print(camera_tensor.shape, camera_tensor.dtype)  # C, H, W

camera_labels = next(
    (group.labels for group in frame.camera_labels if int(group.name) == int(camera.name)),
    [],
)
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(camera_array)
for label in camera_labels:
    box = label.box
    ax.add_patch(
        Rectangle(
            (box.center_x - box.length / 2, box.center_y - box.width / 2),
            box.length,
            box.width,
            fill=False,
            edgecolor="lime",
            linewidth=1,
        )
    )
ax.set_title(f"Camera {camera.name}: {len(camera_labels)} boxes")
ax.axis("off");

## LiDAR point clouds and 3D annotations

`frame.points` contains one first-return XYZ array per laser. Both raw range-image returns remain available under `frame.lasers`.

In [ ]:
points_by_laser = [torch.from_numpy(np.asarray(points)).to(torch.float32) for points in frame.points]
points = torch.cat(points_by_laser, dim=0)
boxes_3d = torch.tensor(
    [
        [
            label.box.center_x, label.box.center_y, label.box.center_z,
            label.box.length, label.box.width, label.box.height, label.box.heading,
        ]
        for label in frame.laser_labels
    ],
    dtype=torch.float32,
)
label_types = torch.tensor([int(label.type) for label in frame.laser_labels], dtype=torch.int64)
print(f"points: {tuple(points.shape)}")
print(f"3D boxes: {tuple(boxes_3d.shape)}; classes: {tuple(label_types.shape)}")

In [ ]:
stride = max(1, len(points) // 30_000)
sample = points[::stride].cpu().numpy()
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(sample[:, 0], sample[:, 1], s=0.2, c=sample[:, 2], cmap="viridis")
if len(boxes_3d):
    ax.scatter(boxes_3d[:, 0], boxes_3d[:, 1], s=12, c="red", label="3D box centers")
    ax.legend()
ax.set(xlabel="vehicle x (m)", ylabel="vehicle y (m)", aspect="equal", title="LiDAR bird's-eye view");

## Range images and segmentation

Parquet conversion exposes decoded range-image arrays directly. Legacy TFRecord caches retain Waymo's compressed payload fields.

In [ ]:
top_laser = next(laser for laser in frame.lasers if int(laser.name) == 1)
first_return = top_laser.ri_return1
if first_return.values is not None:
    range_image = torch.from_numpy(np.array(first_return.values, copy=True)).to(torch.float32)
    camera_projection = torch.from_numpy(np.array(first_return.camera_projection, copy=True)).to(torch.int64)
    valid = range_image[..., 0] > 0
    print("range image:", range_image.shape, "valid pixels:", valid.sum().item())
    print("camera projection:", camera_projection.shape)
else:
    print("Legacy compressed range image bytes:", len(first_return.range_image_compressed))

In [ ]:
segmentation_example = None
for frame_index in range(min(len(dataset), 100)):
    for candidate in dataset[frame_index].images:
        panoptic = candidate.camera_segmentation_label.panoptic_label
        if panoptic is not None and len(panoptic) > 0:
            segmentation_example = (frame_index, candidate, panoptic)
            break
    if segmentation_example is not None:
        break

if segmentation_example is not None:
    frame_index, segmentation_camera, panoptic = segmentation_example
    if isinstance(panoptic, (bytes, bytearray)):
        panoptic_array = np.array(Image.open(io.BytesIO(panoptic)), copy=True)
    else:
        panoptic_array = np.asarray(panoptic)
    panoptic_tensor = torch.from_numpy(panoptic_array.astype(np.int64, copy=False))
    print("frame:", frame_index, "camera:", segmentation_camera.name)
    print("panoptic labels:", panoptic_tensor.shape, panoptic_tensor.dtype)
    plt.figure(figsize=(14, 8))
    plt.imshow(panoptic_tensor.numpy())
    plt.axis("off")
else:
    print("No camera segmentation label found in the first 100 frames; annotations are sparse.")

## Use a transform with `DataLoader`

Frames contain variable-length annotations and point clouds. A transform can select tensors for training; use `batch_size=None` initially, or provide a model-specific padding/collation function.

In [ ]:
def to_torch_sample(item):
    image = np.array(Image.open(io.BytesIO(item.images[0].image)).convert("RGB"), copy=True)
    return {
        "timestamp_micros": item.timestamp_micros,
        "image": torch.from_numpy(image).permute(2, 0, 1),
        "points": torch.cat(
            [torch.from_numpy(np.asarray(value)).to(torch.float32) for value in item.points],
            dim=0,
        ),
        "label_type": torch.tensor([int(label.type) for label in item.laser_labels]),
    }

torch_dataset = WaymoDataset(dataset_root, "training", transform=to_torch_sample)
loader = DataLoader(torch_dataset, batch_size=None, shuffle=False, num_workers=0)
sample = next(iter(loader))
{key: value.shape if isinstance(value, torch.Tensor) else value for key, value in sample.items()}